# 01 — Data Wrangling
## Peramalan Nilai PM2.5 Harian DKI Jakarta dengan SVR

Notebook ini melakukan:
1. Memuat data mentah ISPU DKI Jakarta (`ispu_dki_all.csv`)
2. Memeriksa struktur, satuan, dan periode ketersediaan PM2.5
3. Memilih periode analisis (mulai saat PM2.5 tersedia)
4. Memeriksa dan menangani missing value sesuai aturan di [`docs/dataset.md`](docs/dataset.md)
5. Menyimpan hasil sebagai `data/pm25_daily.csv` untuk dipakai notebook berikutnya (`02_eda.ipynb`)

> **Catatan satuan:** Nilai `pm25` pada dataset ini adalah **angka indeks ISPU**, bukan konsentrasi µg/m³. Lihat `docs/dataset.md` bagian 3 untuk penjelasannya.

## 1. Setup — jalankan di Google Colab

In [ ]:
import os

# Jika dijalankan di Colab dan repo belum ada, clone repo GitHub
if 'google.colab' in str(get_ipython()) and not os.path.exists('data'):
    # Ganti USERNAME dengan username GitHub Anda
    !git clone https://github.com/USERNAME/jakarta-pm25-svr-forecasting.git
    %cd jakarta-pm25-svr-forecasting

print("Direktori kerja saat ini:", os.getcwd())
print("Isi folder data/:", os.listdir('data') if os.path.exists('data') else 'folder data/ belum ada')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 150)

RAW_PATH = 'data/ispu_dki_all.csv'
OUT_PATH = 'data/pm25_daily.csv'

# Batas panjang celah (hari) yang masih boleh diinterpolasi.
# Ini adalah keputusan desain — lihat docs/dataset.md bagian 6 dan
# docs/metodologi.md bagian 3 untuk alasan dan konsekuensinya.
INTERPOLATION_LIMIT = 3

## 2. Memuat Data Mentah

File ini berisi data gabungan dari 5 stasiun pemantau kualitas udara (SPKU) DKI Jakarta. Setiap baris merepresentasikan satu tanggal, dengan kolom `stasiun` menunjukkan stasiun yang nilai ISPU-nya tertinggi pada hari itu (lihat `docs/dataset.md` bagian 5).

In [ ]:
df_raw = pd.read_csv(RAW_PATH, parse_dates=['tanggal'])
df_raw = df_raw.sort_values('tanggal').reset_index(drop=True)

print(f"Jumlah baris   : {len(df_raw):,}")
print(f"Rentang tanggal: {df_raw['tanggal'].min().date()} s.d. {df_raw['tanggal'].max().date()}")
print(f"Kolom          : {list(df_raw.columns)}")
df_raw.head()

In [ ]:
df_raw.info()

## 3. Memeriksa Ketersediaan PM2.5

`pm25` tidak tersedia sepanjang rentang file. Sel di bawah mencari tanggal pertama dan terakhir yang datanya valid (bukan NaN).

In [ ]:
pm25_valid = df_raw.dropna(subset=['pm25'])

tgl_awal = pm25_valid['tanggal'].min()
tgl_akhir = pm25_valid['tanggal'].max()

print(f"Tanggal PM2.5 pertama tersedia : {tgl_awal.date()}")
print(f"Tanggal PM2.5 terakhir tersedia: {tgl_akhir.date()}")
print(f"Jumlah baris dengan pm25 terisi: {len(pm25_valid):,} dari {len(df_raw):,} baris total")

# Persentase pm25 kosong per tahun, untuk melihat sejak kapan datanya konsisten ada
persen_kosong_per_tahun = (
    df_raw.assign(tahun=df_raw['tanggal'].dt.year)
    .groupby('tahun')['pm25']
    .apply(lambda s: s.isna().mean() * 100)
    .round(1)
)
print("\nPersentase pm25 KOSONG per tahun:")
print(persen_kosong_per_tahun)

## 4. Menentukan Periode Analisis

Berdasarkan pemeriksaan di atas, periode analisis dipilih mulai dari tanggal pertama PM2.5 tersedia secara konsisten (biasanya awal tahun setelah tahun dengan data 100% kosong) sampai tanggal terakhir pada file.

In [ ]:
# Tentukan tanggal mulai berdasarkan tahun pertama yang datanya sudah tersedia
tahun_mulai = persen_kosong_per_tahun[persen_kosong_per_tahun < 100].index.min()
TANGGAL_MULAI = pd.Timestamp(year=int(tahun_mulai), month=1, day=1)
TANGGAL_AKHIR = df_raw['tanggal'].max()

print(f"Periode analisis dipakai: {TANGGAL_MULAI.date()} s.d. {TANGGAL_AKHIR.date()}")

df = df_raw[(df_raw['tanggal'] >= TANGGAL_MULAI) & (df_raw['tanggal'] <= TANGGAL_AKHIR)].copy()
df = df.set_index('tanggal')

# Kolom yang dipakai untuk pemodelan; kolom turunan (max, critical, categori)
# dan stasiun tidak disertakan karena bocor informasi atau tidak bisa diprediksi
# (lihat docs/dataset.md bagian 2 dan 5)
KOLOM_DIPAKAI = ['pm25', 'pm10', 'so2', 'co', 'o3', 'no2']
df = df[KOLOM_DIPAKAI]

print(f"Jumlah baris pada periode ini: {len(df):,}")
df.head()

## 5. Memeriksa Tanggal yang Hilang dari Deret Harian

Jumlah baris pada file belum tentu sama dengan jumlah hari kalender pada periode tersebut. Sel berikut menyusun ulang data ke frekuensi harian penuh (`asfreq('D')`) agar tanggal yang hilang terlihat sebagai baris kosong (NaN).

In [ ]:
jumlah_hari_seharusnya = (TANGGAL_AKHIR - TANGGAL_MULAI).days + 1
jumlah_tanggal_ada = df.index.nunique()

print(f"Jumlah hari seharusnya (kalender): {jumlah_hari_seharusnya:,}")
print(f"Jumlah tanggal yang ada di file  : {jumlah_tanggal_ada:,}")
print(f"Jumlah tanggal HILANG            : {jumlah_hari_seharusnya - jumlah_tanggal_ada:,}")
print(f"Jumlah baris duplikat (tanggal sama muncul >1x): {df.index.duplicated().sum()}")

# Susun ulang ke frekuensi harian penuh
df = df[~df.index.duplicated(keep='first')]
df = df.asfreq('D')

print(f"\nSetelah asfreq('D'), jumlah baris: {len(df):,}")

## 6. Memeriksa Missing Value per Kolom

In [ ]:
missing_report = pd.DataFrame({
    'jumlah_NaN': df.isna().sum(),
    'persen_NaN': (df.isna().mean() * 100).round(2)
})
print(missing_report)

# Tanggal spesifik yang pm25-nya kosong (penting dicatat di docs/dataset.md)
tanggal_pm25_kosong = df[df['pm25'].isna()].index
print(f"\nTanggal dengan pm25 kosong ({len(tanggal_pm25_kosong)} hari):")
for t in tanggal_pm25_kosong:
    print(" -", t.date())

## 7. Menganalisis Panjang Celah (Gap) per Kolom

Sebuah kolom bisa punya banyak NaN yang tersebar sebagai celah pendek (1-3 hari), atau terkonsentrasi sebagai celah panjang (berminggu-minggu). Keduanya butuh perlakuan berbeda: celah pendek aman diinterpolasi, celah panjang tidak (lihat `docs/dataset.md` bagian 6).

In [ ]:
def analisis_gap(series, nama_kolom):
    """Mengembalikan daftar panjang celah NaN berturut-turut pada sebuah series."""
    is_na = series.isna()
    if not is_na.any():
        print(f"{nama_kolom}: tidak ada missing value")
        return []
    grup = (is_na != is_na.shift()).cumsum()
    panjang_gap = is_na.groupby(grup).agg(['first', 'size'])
    panjang_gap = panjang_gap[panjang_gap['first']]['size'].tolist()
    panjang_gap.sort(reverse=True)
    terpanjang = panjang_gap[:5]
    print(f"{nama_kolom}: {len(panjang_gap)} celah, total {is_na.sum()} hari NaN, "
          f"5 celah terpanjang: {terpanjang}")
    return panjang_gap

print("Ringkasan panjang celah tiap kolom:\n")
gap_info = {}
for kolom in KOLOM_DIPAKAI:
    gap_info[kolom] = analisis_gap(df[kolom], kolom)

In [ ]:
# Detail celah PANJANG (>= 7 hari) untuk setiap kolom — ini yang TIDAK akan diinterpolasi
print(f"Celah dengan panjang >= 7 hari (di atas ambang INTERPOLATION_LIMIT={INTERPOLATION_LIMIT}):\n")
for kolom in KOLOM_DIPAKAI:
    is_na = df[kolom].isna()
    if not is_na.any():
        continue
    grup = (is_na != is_na.shift()).cumsum()
    for g, sub in df[kolom].groupby(grup):
        if is_na[sub.index].all() and len(sub) >= 7:
            print(f"  {kolom}: {sub.index[0].date()} s.d. {sub.index[-1].date()} "
                  f"({len(sub)} hari)")

## 8. Visualisasi Awal Sebelum Penanganan Missing Value

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.index, df['pm25'], linewidth=0.8)
ax.set_title('Nilai PM2.5 Harian (indeks ISPU) — Sebelum Penanganan Missing Value')
ax.set_ylabel('Nilai ISPU PM2.5')
ax.set_xlabel('Tanggal')
plt.tight_layout()
plt.savefig('docs/pm25_sebelum_cleaning.png', dpi=120)
plt.show()

## 9. Menangani Missing Value

Aturan (lihat `docs/dataset.md` bagian 6 dan `docs/metodologi.md` bagian 3):

- Celah **≤ `INTERPOLATION_LIMIT` hari** diisi dengan interpolasi berbasis waktu.
- Celah **lebih panjang dibiarkan kosong** (tidak diisi), supaya tidak menciptakan data buatan yang tidak realistis.
- Baris yang nantinya butuh fitur lag/rolling yang melewati celah panjang akan dibuang otomatis di notebook `03_feature_engineering.ipynb` (lewat `dropna()`).

In [ ]:
df_bersih = df.copy()

for kolom in KOLOM_DIPAKAI:
    df_bersih[kolom] = df_bersih[kolom].interpolate(
        method='time',
        limit=INTERPOLATION_LIMIT,
        limit_area='inside'
    )

print("Missing value SEBELUM interpolasi:")
print(df.isna().sum())
print("\nMissing value SESUDAH interpolasi (celah > "
      f"{INTERPOLATION_LIMIT} hari tetap kosong):")
print(df_bersih.isna().sum())

## 10. Verifikasi Hasil dan Statistik Deskriptif

In [ ]:
print("Statistik deskriptif nilai PM2.5 (indeks ISPU) setelah pembersihan:\n")
print(df_bersih['pm25'].describe().round(2))

print("\nKorelasi pm25 dengan polutan lain (data yang tidak kosong):")
print(df_bersih[KOLOM_DIPAKAI].corr()['pm25'].round(2).sort_values(ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df_bersih.index, df_bersih['pm25'], linewidth=0.8, color='darkorange')
ax.set_title('Nilai PM2.5 Harian (indeks ISPU) — Setelah Penanganan Missing Value')
ax.set_ylabel('Nilai ISPU PM2.5')
ax.set_xlabel('Tanggal')
plt.tight_layout()
plt.savefig('docs/pm25_sesudah_cleaning.png', dpi=120)
plt.show()

## 11. Ringkasan Kualitas Data (untuk didokumentasikan di `docs/dataset.md`)

Sel ini mencetak semua angka yang perlu disalin ke README/dataset.md, supaya dokumentasi selalu sesuai dengan hasil aktual, bukan angka yang ditempel manual.

In [ ]:
print("=" * 60)
print("RINGKASAN UNTUK docs/dataset.md")
print("=" * 60)
print(f"Periode dipakai       : {TANGGAL_MULAI.date()} s.d. {TANGGAL_AKHIR.date()}")
print(f"Jumlah hari (kalender): {jumlah_hari_seharusnya:,}")
print(f"Tanggal hilang        : {jumlah_hari_seharusnya - jumlah_tanggal_ada:,}")
print(f"pm25 NaN sebelum      : {df['pm25'].isna().sum()}")
print(f"pm25 NaN sesudah      : {df_bersih['pm25'].isna().sum()}")
print(f"Batas interpolasi     : {INTERPOLATION_LIMIT} hari")
print()
for kolom in KOLOM_DIPAKAI:
    n_gap = len(gap_info[kolom])
    total_nan = df[kolom].isna().sum()
    gap_panjang = [g for g in gap_info[kolom] if g > INTERPOLATION_LIMIT]
    print(f"{kolom:6s}: {total_nan:4d} hari NaN dalam {n_gap:3d} celah "
          f"({len(gap_panjang)} celah > {INTERPOLATION_LIMIT} hari)")
print("=" * 60)

## 12. Menyimpan Hasil

Disimpan sebagai `data/pm25_daily.csv`, dipakai sebagai input untuk `02_eda.ipynb`.

In [ ]:
os.makedirs('data', exist_ok=True)
df_bersih.to_csv(OUT_PATH, index=True, index_label='tanggal')
print(f"Tersimpan: {OUT_PATH}  ({len(df_bersih)} baris)")

df_bersih.head()

## Ringkasan

| Langkah | Hasil |
|---|---|
| Sumber | `data/ispu_dki_all.csv` |
| Periode dipakai | Dicetak otomatis pada Sel 4 |
| Baris setelah `asfreq('D')` | Dicetak otomatis pada Sel 5 |
| Missing value pm25 sebelum/sesudah | Dicetak otomatis pada Sel 9/11 |
| Output | `data/pm25_daily.csv` |

**Langkah berikutnya:** buka `02_eda.ipynb` untuk analisis eksploratif (tren, musiman, ACF/PACF).